## *Jeu de données IRVE*
-------------------
# Analyse des relations entre champs (intégrité des données)
------------------------------
## Contexte
- clarification du rôle des modèles de données dans les jeux de données (cf mise à jour récente des [guides data.gouv](https://guides.etalab.gouv.fr/qualite/documenter-les-donnees/))
- intégration d'une propriété "relationship" dans les schémas de données ([issue TableSchema](https://github.com/frictionlessdata/specs/issues/803) en cours de validation)
- création d'outils de contrôle des relations entre champs des jeux de données tabulaires (cf usage ci-dessous)

## Objectifs
- valider sur un cas réel l'utilisation d'un modèle de données en complément d'un schéma de données
- identifier les apports que pourraient avoir les contrôles de validation des relations entre champs

## Résultats
- les principaux résultats sont regroupés sur un [tableau de bord](https://nbviewer.org/github/loco-philippe/Environmental-Sensing/blob/main/python/Validation/irve/Analyse/IRVE_indicateurs.ipynb) 
- la formalisation d'un modèle de données facilite la compréhension des données et des relations entre champs
- l'outil de contrôle permet d'améliorer significativement la qualité des données par l'identification d'incohérences de relations
- l'identification des incohérences permet de trouver des stratégies de réduction des écarts (dans l'exemple ci-dessous, on passe 36% d'écart à 3,8 %)  
- l'analyse des données permet de (re)construire le modèle de données qui minimise les incohérences
- les incohérences détectées sur le jeu de données IRVE restent faibles (inférieures à 5 % des point de charge documentés - voir chapitre 4)

## Suite à donner
- Mettre à jour, valider et publier le modèle de données IRVE
- Définir les contrôles supplémentaires à intégrer pour toutes nouvelles données ainsi que pour le jeu complet
- Mettre en oeuvre les outils de contrôle

## Evolutions possibles 
- Ajouter dans les guides d'Etalab un guide pour les modèles de données 
- Intégrer dans les schémas de données la propriété "relationship" en cours de validation,
- Définir un indicateur qui mesure l'écart (existant / attendu) des relations entre champs

## Sommaire
*(liens actifs sur jupyter Notebook ou Nbviewer)*
- [1 - modèle de données](#1---modèle-de-données)
- [2 - Initialisation](#2---Initialisation)
- [3 - Séparation des pdc itinerance et hors itinerance](#3---Séparation-des-pdc-itinerance-et-hors-itinerance)
- [4 - Bilan initial intégrité](#4---Bilan-initial-intégrité)
- [5 - Séparation doublons pdc - date de maj](#5---Séparation-doublons-pdc---date-de-maj)
- [6 - Séparation doublons station - date de maj](#6---Séparation-doublons-station---date-de-maj)
- [7 - Synthèse](#7---Synthèse)
- [8 - Exemples d'erreurs résiduelles](#8---Exemples-d\'erreurs-résiduelles)

Ce Notebook peut être consulté sur [nbviewer](http://nbviewer.org/github/loco-philippe/Environmental-Sensing/tree/main/python/Validation/irve/Analyse)

données utilisées : https://www.data.gouv.fr/fr/datasets/fichier-consolide-des-bornes-de-recharge-pour-vehicules-electriques/    
fichier : "*consolidation-etalab-schema-irve-statique-v-2.2.0-2024xxxx.csv*"

-----------------------
# 1 - modèle de données

Le modèle de données proposé ci-dessous est construit sur la base du schéma de données mis à disposition et du contenu du jeu de données.    
Il est à consolider en fonction de l'expertise des concepteurs et réutilisateurs (voir [guide méthodologique](https://github.com/loco-philippe/Environmental-Sensing/blob/main/property_relationship/FR_methodology.ipynb)).

*Notation:*
- *M : Mandatory - documentation obligatoire*
- *PK : Primary Key - identifiant unique de l'entité*
- *Root : champ fictif associé à une ligne du tableau*

In [1]:
from base64 import b64encode
from IPython.display import Image, display
with open('IRVE_modele.txt', 'r', encoding="utf-8") as f:
    modele = f.read()
display(Image(url="https://mermaid.ink/img/" + b64encode(modele.encode("ascii")).decode("ascii")))

------
# 2 - Initialisation
## initialisation logicielle

In [2]:
from datetime import datetime
import json
from tab_dataset import Cdataset
import pandas as pd
from pandas import NamedAgg
import ntv_pandas as npd
import pathlib

def analyse_integrite(data, schema, affiche=True, indic=True):
    '''analyse les relations du DataFrame 'data' définies dans le schéma 'schema'.
    Le nombre de ligne en erreur par relation (dict) est retourné et optionnellement affiché (paramètre 'affiche=True') . 
    Les lignes en erreur sont optionnellement ajoutées (paramètre 'indic=True') à 'data' sous forme de champs booléens par relation.
    '''
    dic_errors = Cdataset(data).check_relationship(schema)
    dic_count = {name: len(errors) for name, errors in dic_errors.items()}
    if affiche:
        for name, total in dic_count.items():
            print('{:<50} {:>5}'.format(name, total))
    if indic:
        data['ok'] = True
        for name, errors in dic_errors.items():
            data[name] = True
            data.loc[errors, name] = False
            data['ok'] = data['ok'] & data[name] 
    return dic_count

def add_nbre_pdc(data, affiche=True):
    '''ajoute un champ avec le nombre de pdc calculés et le compare au champ 'nbre_pdc'
    Le nombre d'écarts est affiché optionnellement (paramètre 'affiche=True').
    '''
    data['nb_pdc_calc'] = data.groupby('id_station_itinerance')['index'].transform('count')
    nbre_pdc = data['nbre_pdc'].astype('int')
    data['nbre_pdc_ok'] = data['nb_pdc_calc'] == nbre_pdc
    data['calc-nbre'] = data['nb_pdc_calc'] - nbre_pdc
    if affiche:
        ecart_nbre = len(data) - data['nbre_pdc_ok'].sum()
        print('{:<20} {:>5}'.format('nbre_pdc_ko', ecart_nbre))
        print(round(ecart_nbre / len(data) * 100), ' %')

## initialisation des données
- lecture du fichier issu de l'api 190 000 lignes (120 000 fin 2024)

In [3]:
dates = ['2023-03-03', '2023-04-17', '2023-05-02', '2023-05-24', '2023-06-17', '2023-07-04', '2023-07-23', '2023-08-08',
         '2023-09-06', '2023-09-19', '2023-10-04', '2023-10-21', '2023-10-31', '2023-11-23', '2023-12-14', '2023-12-29',
         '2024-01-15', '2024-02-07', '2024-02-28', '2024-03-27', '2024-04-24', '2024-05-19', '2024-06-05', '2024-08-02',
         '2024-10-24', '2024-11-27', '2025-01-03', '2025-10-04', '2025-11-18', '2026-02-18', '2026-09-29']
date= dates[-1]
log = {'date_irve': date, 
    'file': 'consolidation-etalab-schema-irve-statique-v-2.3.1-'+date[:4]+date[5:7]+date[8:]+'.csv', # après 2024-02-06
    'chemin': str(pathlib.Path(npd.__file__).parent.parent.parent/"Environmental-Sensing"/"python"/"Validation"/"irve"/"Analyse")
      }
irve = pd.read_csv(log['chemin'] + '/' + log['file'], sep=',', low_memory=False, dtype='object')
log['len_irve'] = len(irve)
print('nombre de lignes : ', log['len_irve']) 

nombre de lignes :  169404


## schéma de données
Le schéma de données restreint à la propriété 'relationship' et construit à partir du modèle de données est le suivants :

In [4]:
# complément à inclure dans le schéma de données
schema = {
    'relationships': [
         # relation unicité des pdl
         {"fields": ["id_pdc_itinerance", "index"],                    "link" : "coupled" },   
         # relations inter entités
         {"fields": ["id_station_itinerance", "contact_operateur"],    "link" : "derived" },
         {"fields": ["id_station_itinerance", "nom_enseigne"],         "link" : "derived" },
         {"fields": ["id_station_itinerance", "coordonneesXY"],        "link" : "derived" },
         {"fields": ["id_pdc_itinerance", "id_station_itinerance"],    "link" : "derived" },
         # relations intra entité - station
         {"fields": ["id_station_itinerance", "nom_station"],          "link" : "derived" },
         {"fields": ["id_station_itinerance", "implantation_station"], "link" : "derived" },
         #{"fields": ["id_station_itinerance", "date_maj"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "nbre_pdc"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "condition_acces"],      "link" : "derived" },
         {"fields": ["id_station_itinerance", "horaires"],             "link" : "derived" },
         {"fields": ["id_station_itinerance", "station_deux_roues"],   "link" : "derived" },
         # relations intra entité - localisation
         {"fields": ["coordonneesXY", "adresse_station"],              "link" : "derived" }
    ]
}
# liste des champs liés à un controle (relations) et obligatoires (mandatory)
relations = ['index', 'contact_operateur', 'nom_enseigne', 'coordonneesXY', 'adresse_station', 'id_station_itinerance', 
             'nom_station', 'implantation_station', 'nbre_pdc', 'condition_acces', 'horaires', 'station_deux_roues', 
             'id_pdc_itinerance', 'date_maj', 'last_modified', 'datagouv_organization_or_owner']
mandatory = ['contact_operateur', 'nom_enseigne', 'coordonneesXY', 'adresse_station', 'id_station_itinerance', 'nom_station',
             'implantation_station', 'nbre_pdc', 'condition_acces', 'horaires', 'station_deux_roues', 'id_pdc_itinerance', 
             'puissance_nominale', 'prise_type_ef', 'prise_type_2', 'prise_type_combo_ccs', 'prise_type_chademo', 
             'prise_type_autre', 'paiement_acte', 'paiement_autre', 'reservation',  'accessibilite_pmr', 'restriction_gabarit', 
             'date_maj', 'last_modified']
controles = ['index - id_pdc_itinerance', 'contact_operateur - id_station_itinerance', 'nom_enseigne - id_station_itinerance',
             'coordonneesXY - id_station_itinerance', 'id_station_itinerance - id_pdc_itinerance',
             'nom_station - id_station_itinerance', 'implantation_station - id_station_itinerance',
             'nbre_pdc - id_station_itinerance', 'condition_acces - id_station_itinerance', 'horaires - id_station_itinerance',
             'station_deux_roues - id_station_itinerance', 'adresse_station - coordonneesXY']
origine = 'datagouv_organization_or_owner'

-----------------------
## 3 - Séparation des pdc itinerance et hors itinerance
- 0.14 % des points de recharge sont hors itinerance (1 % en 2024)

In [5]:
# séparation des données
data = irve
data[['id_station_itinerance','id_pdc_itinerance','restriction_gabarit']] = data[['id_station_itinerance','id_pdc_itinerance','restriction_gabarit']].astype('string')
if 'last_modified' not in data.columns:
    data['last_modified'] = data['date_maj']
#data[['last_modified','date_maj']] = data[['last_modified','date_maj']].astype('datetime64')
data['non_concerne'] = data['id_station_itinerance'].str.contains('oncern') | data['id_pdc_itinerance'].str.contains('oncern')

non_concerne = data[data['non_concerne']].reset_index()['index']
itinerance = data[~data['non_concerne']].reset_index()
itinerance_init = itinerance.loc[:, relations]
log['pdc_hors_itinerance'] = len(non_concerne)
log['pdc_en_itinerance'] = len(itinerance)
print('nombre de pdc hors itinerance : ', log['pdc_hors_itinerance'], 'soit : ', len(non_concerne)/(len(non_concerne)+len(itinerance)))
print('nombre de pdc en itinerance   : ', log['pdc_en_itinerance'])

nombre de pdc hors itinerance :  383 soit :  0.0022608675119831882
nombre de pdc en itinerance   :  169021


In [6]:
# arborescence des champs et nombre de valeurs différentes
print(itinerance_init.npd.analysis().tree())

-1: root-derived (169021)
   0 : index (0 - 169021)
   12: id_pdc_itinerance (2 - 169019)
      1 : contact_operateur (168770 - 249)
      2 : nom_enseigne (162820 - 6199)
      3 : coordonneesXY (130208 - 38811)
      4 : adresse_station (129070 - 39949)
      5 : id_station_itineranc (119630 - 49389)
      6 : nom_station (128035 - 40984)
      7 : implantation_station (169014 - 5)
      8 : nbre_pdc (168907 - 112)
      9 : condition_acces (169017 - 2)
      10: horaires (167706 - 1313)
      11: station_deux_roues (169017 - 2)
      13: date_maj (167955 - 1064)
         14: last_modified (0 - 1064)
      15: datagouv_organizatio (168869 - 150)


-----------------------
## 4 - Bilan initial intégrité
- 58 % des lignes présentent un défaut d'intégrité (moins de 20% en 2024)
- 9 % avec le nouveau fichier transport

In [7]:
# séparation données bonnes (itinerance_ok_1) et données résiduelles (itinerance_1)
res = analyse_integrite(itinerance_init, schema)
itinerance_ok_1 = itinerance_init.loc[itinerance_init.ok, relations].reset_index(drop=True)
itinerance_1 = itinerance_init.loc[~itinerance_init.ok, relations].reset_index(drop=True)
itinerance_init = itinerance_init.loc[:, relations]
log['init_ok'] = len(itinerance_ok_1)
log['init_ko'] = len(itinerance_1)
print("\nnombre d'enregistrements sans erreurs : ", log['init_ok'])
print("nombre d'enregistrements avec au moins une erreur : ", log['init_ko'])
print("taux d'erreur : ", round(log['init_ko'] / log['pdc_en_itinerance'] * 100), ' %')

index - id_pdc_itinerance                              4
contact_operateur - id_station_itinerance           1101
nom_enseigne - id_station_itinerance                5154
coordonneesXY - id_station_itinerance               1910
id_station_itinerance - id_pdc_itinerance              0
nom_station - id_station_itinerance                  998
implantation_station - id_station_itinerance        1136
nbre_pdc - id_station_itinerance                    5905
condition_acces - id_station_itinerance               85
horaires - id_station_itinerance                     486
station_deux_roues - id_station_itinerance           297
adresse_station - coordonneesXY                    13130

nombre d'enregistrements sans erreurs :  153353
nombre d'enregistrements avec au moins une erreur :  15668
taux d'erreur :  9  %


-----------------------
## 5 - Séparation doublons pdc - date de maj
- 51 % des pdc en erreur sont liées aux doublons de pdc (40 % fin 2024)
- la suppression des doublons permet de réduire de 94% le nombre de lignes erronnées (50 % fin 2024)

- présence uniquement de 2 doublons avec le nouveau fichier transport

In [8]:
# séparation doublons pdc (doublons_pdc) et données résiduelles (itinerance_2)
itinerance_1['doublons_pdc'] = itinerance_1.sort_values(by=['date_maj', 'last_modified']).duplicated('id_pdc_itinerance', keep='last')

doublons_pdc = itinerance_1[itinerance_1['doublons_pdc']].loc[:, relations].reset_index(drop=True)['index']
#doublons_pdc = itinerance_1[itinerance_1['doublons_pdc']].loc[:, relations].reset_index(drop=True)
itinerance_2 = itinerance_1[~itinerance_1['doublons_pdc']].loc[:, relations].reset_index(drop=True)
itinerance_1 = itinerance_1.loc[:, relations]
log['doublons_pdc'] = len(doublons_pdc)
log['sans_doublons_pdc'] = len(itinerance_2)
print('nombre de doublons pdc : ', log['doublons_pdc'], ' soit ', round(log['doublons_pdc']/log['init_ko'] * 100), ' %')
print('nombre de pdc sans doublon   : ', log['sans_doublons_pdc'], ' soit ', round(log['sans_doublons_pdc']/log['init_ko'] * 100), ' %')

nombre de doublons pdc :  2  soit  0  %
nombre de pdc sans doublon   :  15666  soit  100  %


In [9]:
# séparation données bonnes (itinerance_ok_3) et données résiduelles (itinerance_3)
res = analyse_integrite(itinerance_2, schema)
itinerance_ok_3 = itinerance_2.loc[itinerance_2.ok, relations].reset_index(drop=True)
itinerance_3 = itinerance_2.loc[~itinerance_2.ok, relations].reset_index(drop=True)
itinerance_2 = itinerance_2.loc[:, relations]
log['etape3_ok'] = len(itinerance_ok_3)
log['etape3_ko'] = len(itinerance_3)
print("\nnombre d'enregistrements sans erreurs : ", log['etape3_ok'])
print("nombre d'enregistrements avec au moins une erreur : ", log['etape3_ko'])

index - id_pdc_itinerance                              0
contact_operateur - id_station_itinerance           1101
nom_enseigne - id_station_itinerance                5154
coordonneesXY - id_station_itinerance               1910
id_station_itinerance - id_pdc_itinerance              0
nom_station - id_station_itinerance                  998
implantation_station - id_station_itinerance        1136
nbre_pdc - id_station_itinerance                    5905
condition_acces - id_station_itinerance               85
horaires - id_station_itinerance                     486
station_deux_roues - id_station_itinerance           297
adresse_station - coordonneesXY                    13130

nombre d'enregistrements sans erreurs :  2
nombre d'enregistrements avec au moins une erreur :  15664


-----------------------
## 6 - Séparation doublons station - date de maj
- la suppression des anciens pdc permet de réduire de 13 % supplémentaire le nombre de lignes erronnées
- les dernières erreurs correspondent à des stations associées à 69 opérateurs et sont liées à des causes multiples

In [10]:
# séparation doublons stations (doublons_stat_maj) et données résiduelles (itinerance_4)
itinerance_3['stat_maj'] = itinerance_3.id_station_itinerance + itinerance_3.date_maj
stat_maj_unique = itinerance_3.sort_values(by='stat_maj').drop_duplicates('id_station_itinerance', keep='last')
itinerance_3['last_stat_maj'] = itinerance_3['stat_maj'].isin(stat_maj_unique['stat_maj'])

doublons_stat_maj = itinerance_3[~itinerance_3['last_stat_maj']].loc[:, relations].reset_index(drop=True)['index']
itinerance_4 = itinerance_3[itinerance_3['last_stat_maj']].loc[:, relations].reset_index(drop=True)
itinerance_3 = itinerance_3.loc[:, relations]
log['doublons_station'] = len(doublons_stat_maj)
log['sans_doublons_station'] = len(itinerance_4)
print('nombre de doublons stations : ', log['doublons_station'], ' soit ', 
      round(log['doublons_station']/log['init_ko'] * 100), ' %')
print('nombre de pdc sans doublon   : ', log['sans_doublons_station'])

nombre de doublons stations :  695  soit  4  %
nombre de pdc sans doublon   :  14969


In [11]:
# séparation données bonnes (itinerance_ok_5) et données résiduelles (itinerance_5 / itinerance_5_full)
res = analyse_integrite(itinerance_4, schema)
itinerance_ok_5 = itinerance_4.loc[itinerance_4.ok, relations].reset_index(drop=True)
itinerance_5_full = itinerance_4.loc[~itinerance_4.ok].reset_index(drop=True)
itinerance_5 = itinerance_5_full.loc[:, relations]
#itinerance_4 = itinerance_4.loc[:, relations]
log['etape5_ok'] = len(itinerance_ok_5)
log['etape5_ko'] = len(itinerance_5)
print("\nnombre d'enregistrements sans erreurs : ", log['etape5_ok'])
print("nombre d'enregistrements avec au moins une erreur : ", log['etape5_ko'], ' soit ', 
      round(log['etape5_ko']/log['etape3_ko'] * 100), ' %')

index - id_pdc_itinerance                              0
contact_operateur - id_station_itinerance              2
nom_enseigne - id_station_itinerance                4100
coordonneesXY - id_station_itinerance               1009
id_station_itinerance - id_pdc_itinerance              0
nom_station - id_station_itinerance                  497
implantation_station - id_station_itinerance         130
nbre_pdc - id_station_itinerance                    4496
condition_acces - id_station_itinerance                0
horaires - id_station_itinerance                     112
station_deux_roues - id_station_itinerance             0
adresse_station - coordonneesXY                    12582

nombre d'enregistrements sans erreurs :  1012
nombre d'enregistrements avec au moins une erreur :  13957  soit  89  %


In [12]:
# structure des données présentant une erreur et nombre de valeurs
print(itinerance_5.npd.analysis().tree())

-1: root-derived (13957)
   0 : index (0 - 13957)
   1 : contact_operateur (13860 - 97)
   2 : nom_enseigne (12535 - 1422)
   3 : coordonneesXY (11392 - 2565)
   4 : adresse_station (9348 - 4609)
   5 : id_station_itineranc (10058 - 3899)
      9 : condition_acces (3897 - 2)
      11: station_deux_roues (3897 - 2)
      13: date_maj (3571 - 328)
         14: last_modified (0 - 328)
   6 : nom_station (10631 - 3326)
   7 : implantation_station (13952 - 5)
   8 : nbre_pdc (13909 - 48)
   10: horaires (13795 - 162)
   12: id_pdc_itinerance (0 - 13957)
   15: datagouv_organizatio (13894 - 63)


In [13]:
# nb maxi d'erreurs
itinerance_4['somme'] = 0
for name in res.keys():
    itinerance_4['somme'] += 1 - itinerance_4[name]
erreurs = max(itinerance_4['somme'])
maxi = itinerance_4[itinerance_4.somme >= erreurs-1]
print("nombre d'enregistrements avec ", erreurs-1, "(ou plus)  erreurs : ", len(maxi))

nombre d'enregistrements avec  4 (ou plus)  erreurs :  61


In [14]:
# principaux opérateurs contributeur de défauts
operateurs = list(itinerance_5.drop_duplicates('contact_operateur')['contact_operateur'])
erreurs_op = [len(itinerance_5.loc[itinerance_5.contact_operateur == operateur]) for operateur in operateurs]
err_op_5, oper_5 = tuple(zip(*(sorted(zip(erreurs_op, operateurs), reverse=True))[:5]))
err_op_10, oper_10 = tuple(zip(*(sorted(zip(erreurs_op, operateurs), reverse=True))[:10]))

print(len(err_op_5), ' opérateurs représentent : ', round(sum(err_op_5) / sum(erreurs_op) * 100), ' % des écarts\n')
for err, op in zip(err_op_5, oper_5):
    print(err, op)

5  opérateurs représentent :  60  % des écarts

4496 sav@izivia.com
1396 supervision-ev.france@totalenergies.com
1139 support@alizecharge.fr
819 info@ionity.eu
470 contact@qovoltis.com


In [15]:
print(len(err_op_10), ' opérateurs représentent : ', round(sum(err_op_10) / sum(erreurs_op) * 100), ' % des écarts\n')
for err, op in zip(err_op_10, oper_10):
    print(err, op)

10  opérateurs représentent :  73  % des écarts

4496 sav@izivia.com
1396 supervision-ev.france@totalenergies.com
1139 support@alizecharge.fr
819 info@ionity.eu
470 contact@qovoltis.com
462 assistance-technique@ouestcharge-paysdelaloire-moncompte.fr
437 roaming@zunder.com
401 fabien.lepeytre@allego.eu
276 kg@lastmilesolutions.com
250 bornes@lidl.fr


-----------------------
## 7 - Synthèse


### fichiers
Génération des fichiers intégrant les défauts d'intégrité :
- fichier csv des lignes résiduelles à traiter (IRVE_itinerance_residuel)
- fichier csv des données itinerance avec indicateur des données à corriger ou à ignorer (IRVE_itinerance_complet)
- fichier csv des données itinerance valides (IRVE_itinerance_valide)
- fichier csv des doublons (IRVE_itinerance_doublons)

In [52]:
# consolidation des données
itinerance['doublons_stat_maj'] = itinerance['index'].isin(doublons_stat_maj)
itinerance['doublons_pdc'] = itinerance['index'].isin(doublons_pdc)
itinerance['lignes_a_corriger'] = itinerance['index'].isin(itinerance_5['index'])
itinerance['doublons_a_supprimer'] = itinerance['doublons_stat_maj'] | itinerance['doublons_pdc']
itinerance['lignes_ko'] = itinerance['doublons_a_supprimer'] | itinerance['lignes_a_corriger']
print('total des lignes : ', len(itinerance))
print('total des lignes à corriger : ', itinerance['lignes_a_corriger'].sum())
itinerance_doublons = itinerance[itinerance['doublons_a_supprimer']].reset_index(drop=True)
print('total des doublons à supprimer : ', len(itinerance_doublons))
itinerance_ok = itinerance[~itinerance['lignes_ko']].reset_index(drop=True)
print('nombre de pdc avec controles ok : ', len(itinerance_ok))

total des lignes :  169021
total des lignes à corriger :  13957
total des doublons à supprimer :  697
nombre de pdc avec controles ok :  154367


In [17]:
# génération des fichiers
extension = log['date_irve'] +'.csv'
itinerance_5_full.to_csv('IRVE_itinerance_residuel' + extension)
itinerance.to_csv('IRVE_itinerance_complet' + extension)
itinerance_ok.to_csv('IRVE_itinerance_valide' + extension)
itinerance_doublons.to_csv('IRVE_itinerance_doublons' + extension)
log['IRVE_itinerance_residuel' + extension] = len(itinerance_5_full)
log['IRVE_itinerance_complet' + extension] = len(itinerance)
log['IRVE_itinerance_valide' + extension] = len(itinerance_ok)
log['IRVE_itinerance_valide_stat' + extension] = len(itinerance_ok.drop_duplicates('id_station_itinerance', keep='last'))
log['IRVE_itinerance_doublons' + extension] = len(itinerance_doublons)

In [18]:
l_prefix = list(pd.read_csv(log['chemin'] + '/' + 'Operations_Recharge_16-07-2024.csv', sep=';', low_memory=False, dtype='object')['Identifiant'])
l_prefix[0]

'FR073'

In [19]:
for op in oper_5:
    print('IRVE_itinerance_residuel' + '_' + op + '_' + extension)
    itinerance_5_full.loc[itinerance_5_full.contact_operateur == op].to_csv('IRVE_itinerance_residuel' + '_' + op + '_' + extension)

IRVE_itinerance_residuel_sav@izivia.com_2026-09-29.csv
IRVE_itinerance_residuel_supervision-ev.france@totalenergies.com_2026-09-29.csv
IRVE_itinerance_residuel_support@alizecharge.fr_2026-09-29.csv
IRVE_itinerance_residuel_info@ionity.eu_2026-09-29.csv
IRVE_itinerance_residuel_contact@qovoltis.com_2026-09-29.csv


### vérification de l'intégrité

In [20]:
# vérification de l'absence d'erreurs
res = analyse_integrite(itinerance_ok.loc[:, relations], schema, affiche=False)
log['bilan_erreurs'] = sum(res.values())
log['date'] = datetime.now().isoformat()
print('bilan intégrité :')
print('    erreurs : ', log['bilan_erreurs'])

bilan intégrité :
    erreurs :  0


In [21]:
# structure des données bonnes
#print(Cdataset(itinerance_ok.loc[:, mandatory]).tree())
print(itinerance_ok.loc[:, mandatory].npd.analysis().tree())

-1: root-derived (154367)
   4 : id_station_itineranc (108877 - 45490)
      0 : contact_operateur (45250 - 240)
      1 : nom_enseigne (40609 - 4881)
      2 : coordonneesXY (9276 - 36214)
         3 : adresse_station (833 - 35381)
      5 : nom_station (7828 - 37662)
      6 : implantation_station (45485 - 5)
      7 : nbre_pdc (45385 - 105)
      8 : condition_acces (45488 - 2)
      9 : horaires (44302 - 1188)
      10: station_deux_roues (45488 - 2)
   11: id_pdc_itinerance (0 - 154367)
   12: puissance_nominale (154126 - 241)
   13: prise_type_ef (154365 - 2)
   14: prise_type_2 (154365 - 2)
   15: prise_type_combo_ccs (154365 - 2)
   16: prise_type_chademo (154365 - 2)
   17: prise_type_autre (154365 - 2)
   18: paiement_acte (154365 - 2)
   19: paiement_autre (154364 - 3)
   20: reservation (154365 - 2)
   21: accessibilite_pmr (154363 - 4)
   22: restriction_gabarit (154210 - 157)
   23: date_maj (153355 - 1012)
      24: last_modified (0 - 1012)


### Structure des données valides

In [22]:
quali = 'QualiCharge'
print(len(itinerance[itinerance[origine]==quali]), len(itinerance_1[itinerance_1[origine]==quali]),
      len(itinerance_2[itinerance_2[origine]==quali]), len(itinerance_3[itinerance_3[origine]==quali]), 
      len(itinerance_4[itinerance_4[origine]==quali]), len(itinerance_5[itinerance_5[origine]==quali]))

78726 6372 6372 6372 6295 5484


In [ ]:
stat_ok = itinerance_ok.groupby(origine).agg(nb_pdcs=NamedAgg("id_pdc_itinerance", "count")).reset_index()
print(len(itinerance), len(itinerance_ok), len(itinerance[itinerance[origine]==quali]), len(itinerance_ok[itinerance_ok[origine]==quali]))
stat_ok.sort_values(by='nb_pdcs', ascending=False)[0:30]

In [55]:
itinerance_ok['index']

0              0
1              1
2              2
3              3
4              4
           ...  
154362    169399
154363    169400
154364    169401
154365    169402
154366    169403
Name: index, Length: 154367, dtype: int64

In [57]:
itinerance_ok_qualicharge = itinerance_ok.loc[itinerance_ok[origine] == quali, :]
stations_qualicharge = itinerance_ok['id_station_itinerance'].isin(itinerance_ok_qualicharge['id_station_itinerance'])
itinerance_ok_additionnel = itinerance_ok.loc[~stations_qualicharge, :]
itinerance_ok_ko = itinerance_ok.loc[stations_qualicharge & (itinerance_ok[origine] != quali), :]
index_ok = list(itinerance_ok_qualicharge['index']) + list(itinerance_ok_additionnel['index'])
itinerance_ko = itinerance[~itinerance['index'].isin(index_ok)]

print(len(itinerance_ok), len(itinerance_ok_qualicharge), len(itinerance_ok_additionnel), len(itinerance_ok_ko))
print(len(itinerance), len(itinerance_ok_qualicharge), len(itinerance_ok_additionnel), len(itinerance_ko))

154367 73165 81174 28
169021 73165 81174 14682


In [63]:
sum(analyse_integrite(itinerance_ok_qualicharge.loc[:, relations], schema, affiche=False).values())

0

In [31]:
# vérification de l'absence d'erreurs
res = analyse_integrite(itinerance_ok_additionnel.loc[:, relations], schema, affiche=False)
log['bilan_erreurs'] = sum(res.values())
log['date'] = datetime.now().isoformat()
print('bilan intégrité :')
print('    erreurs : ', log['bilan_erreurs'], '\n')
print(itinerance_ok_additionnel.loc[:, mandatory].npd.analysis().tree())

bilan intégrité :
    erreurs :  0 

-1: root-derived (81174)
   4 : id_station_itineranc (51467 - 29707)
      0 : contact_operateur (29506 - 201)
      1 : nom_enseigne (27183 - 2524)
      2 : coordonneesXY (8419 - 21288)
         3 : adresse_station (564 - 20724)
      5 : nom_station (7213 - 22494)
      6 : implantation_station (29702 - 5)
      7 : nbre_pdc (29620 - 87)
      8 : condition_acces (29705 - 2)
      9 : horaires (28660 - 1047)
      10: station_deux_roues (29705 - 2)
      20: reservation (29705 - 2)
      22: restriction_gabarit (29577 - 130)
   11: id_pdc_itinerance (0 - 81174)
   12: puissance_nominale (81016 - 158)
   13: prise_type_ef (81172 - 2)
   14: prise_type_2 (81172 - 2)
   15: prise_type_combo_ccs (81172 - 2)
   16: prise_type_chademo (81172 - 2)
   17: prise_type_autre (81172 - 2)
   18: paiement_acte (81172 - 2)
   19: paiement_autre (81171 - 3)
   21: accessibilite_pmr (81170 - 4)
   23: date_maj (80246 - 928)
      24: last_modified (0 - 928)


In [59]:
itinerance_ok_additionnel.to_parquet("IRVE_hors_qualicharge.parquet", index=False)
itinerance_ko.to_parquet("IRVE_hors_qualicharge_non_integre.parquet", index=False)
print("IRVE_hors_qualicharge_integre : ", len(itinerance_ok_additionnel))
print("IRVE_hors_qualicharge_non_integre : ", len(itinerance_ko))

IRVE_hors_qualicharge_integre :  81174
IRVE_hors_qualicharge_non_integre :  14682


### Indicateurs

In [ ]:
# génération des indicateurs liés aux règles
regles = ['Pdc non unique', 'Station multi-operateurs', 'Station multi-enseignes', 'Station multi-localisations', 
          'Pdc multi-stations', 'station avec plusieurs noms', 'station multi-implantations', 
          'nombre de pdc par station incoherent', 'station multi-acces', 'station multi-horaires', 
          'acces deux-roues incoherent', 'localisation multi-adresses']
principal = [16, 17, 18, 19, 20]
secondaire = [21, 22, 23, 24, 25, 26, 27]
irve = itinerance_5_full # residuel
total = len(irve)
indic = {}

irve['principal'] = True
for ind in principal:
    irve['principal'] &= irve.iloc[:,ind]
    indic[regles[ind-16]] = int(total - irve.iloc[:,ind].sum())
irve['secondaire'] = True
for ind in secondaire:
    irve['secondaire'] &= irve.iloc[:,ind]
    indic[regles[ind-16]] = int(total - irve.iloc[:,ind].sum())
irve['secondaire'] |= (~irve['principal'] & ~irve['secondaire'])
irve['verif'] = irve['principal'] & irve['secondaire']
indic['principal pdc'] = int(total - irve['principal'].sum())
indic['secondaire pdc'] = int(total - irve['secondaire'].sum())

irve_p = irve[~irve['principal']].drop_duplicates('id_station_itinerance').reset_index(drop=True)
irve_s = irve[~irve['secondaire']].drop_duplicates('id_station_itinerance').reset_index(drop=True)
indic['principal stat'] = len(irve_p)
indic['secondaire stat'] = len(irve_s)

In [ ]:
# ajout des principaux opérateurs contributeur de défauts
indic['operateurs'] = list(oper_5)
indic['erreurs_operateurs'] = list(err_op_5)

In [ ]:
# ajout de l'indicateur d'écart entre nombre de pdc calculés et saisis
add_nbre_pdc(irve, affiche=True)
indic['nb_pdc_calc-nbre_pdc'] = int(irve['calc-nbre'].sum())

In [ ]:
# indicateur d'écart entre nombre de pdc calculés et saisis sur les données sans doublon
itinerance_sans_doublons = itinerance[~itinerance['doublons_a_supprimer']].reset_index(drop=True)
add_nbre_pdc(itinerance_sans_doublons, affiche=True)

In [ ]:
len(itinerance_4)

In [ ]:
# stockage des indicateurs
log |= indic
log_file = log['chemin'] + '/' + 'logfile.txt'
#with open('logfile.txt', 'a', encoding="utf-8") as f:
with open(log_file, 'a', encoding="utf-8") as f:
    f.write(json.dumps(log) + '\n')

In [ ]:
itinerance_5_full.columns[16:28]

-----------------------
## 8 - Exemples d'erreurs résiduelles 

### Erreurs multiples 
- exemple avec le nombre maximal d'erreurs (4) (35 pdc)
- exemple station 'FRS35P35093001B1' avec 25 pdc : coordonnées XY, nom_station, implantation station, condition accès)

In [39]:
maxi.loc[:, relations][0:60]

,index,contact_operateur,nom_enseigne,coordonneesXY,adresse_station,id_station_itinerance,nom_station,implantation_station,nbre_pdc,condition_acces,horaires,station_deux_roues,id_pdc_itinerance,date_maj,last_modified,datagouv_organization_or_owner
548,7322,sav@izivia.com,SIGEIF,"[2.51143, 48.74988]",30 Rue de Paris 94470 Boissy-Saint-Léger,FRSIGPSIGE519,"SIGEIF - Rue de Paris, Boissy-Saint-Léger",Voirie,2,Accès libre,24/7,false,FRSIGESIGE51912,2026-09-27,2026-09-27,IZIVIA
1442,15216,sav@izivia.com,Izivia Express,"[4.86711, 45.63320]",2 Rue de la Piscine 69360 Saint-Symphorien-d'Ozon,FRE04POAZS340,SIGERLY - SSO03,Voirie,2,Accès libre,24/7,false,FRE04EOAZS34012,2026-09-27,2026-09-27,IZIVIA
3383,20929,sav@izivia.com,MAMP,"[5.29714, 43.42008]",2 Chemin de la Capelane 13170 Les Pennes-Mirabeau,FRIZMPMAMP82,IZIVIA MAMP | Les-Pennes-Mirabeau - Rue Capelane,Voirie,2,Accès libre,24/7,false,FRIZMEMAMP8212,2026-09-27,2026-09-27,IZIVIA
3384,20930,sav@izivia.com,MAMP,"[5.08696, 43.70215]",2 Route de la Tuiliere 13113 Lamanon,FRIZMPMAMP83,IZIVIA MAMP | Lamanon - Route de la Tuiliere,Voirie,2,Accès libre,24/7,false,FRIZMEMAMP8312,2026-09-27,2026-09-27,IZIVIA
4567,55295,bornes@lidl.fr,LIDL,"[2.33401, 48.63200]",rue du Plessis,FRLDLPLFR4256EVCP,Saint-Geneviève-des-Bois - Plessis,Parking privé à usage public,8,Accès libre,24/7,false,FRLDLE00003929,2025-12-11,2025-12-11,Lidl
4568,55296,bornes@lidl.fr,LIDL,"[2.33401, 48.63200]",rue du Plessis,FRLDLPLFR4256EVCP,Saint-Geneviève-des-Bois - Plessis,Parking privé à usage public,8,Accès libre,24/7,false,FRLDLE00003930,2025-12-11,2025-12-11,Lidl
4569,55297,bornes@lidl.fr,LIDL,"[2.33401, 48.63200]",rue du Plessis,FRLDLPLFR4256EVCP,Saint-Geneviève-des-Bois - Plessis,Parking privé à usage public,8,Accès libre,24/7,false,FRLDLE00003931,2025-12-11,2025-12-11,Lidl
4570,55298,bornes@lidl.fr,LIDL,"[2.33401, 48.63200]",rue du Plessis,FRLDLPLFR4256EVCP,Saint-Geneviève-des-Bois - Plessis,Parking privé à usage public,8,Accès libre,24/7,false,FRLDLE00003932,2025-12-11,2025-12-11,Lidl
4571,55303,bornes@lidl.fr,LIDL,"[2.33401, 48.63200]",rue du Plessis,FRLDLPLFR4256EVCP,Saint-Geneviève-des-Bois - Plessis,Parking privé à usage public,8,Accès libre,24/7,false,FRLDLE00003963,2025-12-11,2025-12-11,Lidl
4572,55304,bornes@lidl.fr,LIDL,"[2.33401, 48.63200]",rue du Plessis,FRLDLPLFR4256EVCP,Saint-Geneviève-des-Bois - Plessis,Parking privé à usage public,8,Accès libre,24/7,false,FRLDLE00003964,2025-12-11,2025-12-11,Lidl


In [41]:
itinerance.loc[itinerance.id_station_itinerance == 'FRSIGPSIGE519', relations]

,index,contact_operateur,nom_enseigne,coordonneesXY,adresse_station,id_station_itinerance,nom_station,implantation_station,nbre_pdc,condition_acces,horaires,station_deux_roues,id_pdc_itinerance,date_maj,last_modified,datagouv_organization_or_owner
7322,7322,sav@izivia.com,SIGEIF,"[2.51143, 48.74988]",30 Rue de Paris 94470 Boissy-Saint-Léger,FRSIGPSIGE519,"SIGEIF - Rue de Paris, Boissy-Saint-Léger",Voirie,2,Accès libre,24/7,false,FRSIGESIGE51912,2026-09-27,2026-09-27,IZIVIA
135155,135536,sav@izivia.com,"SIGEIF - Rue de Paris, Boissy-Saint-Léger","[2.51143, 48.74988]","30 Rue de Paris, 94470 Boissy-Saint-Léger",FRSIGPSIGE519,"SIGEIF - Rue de Paris, Boissy-Saint-Léger",Voirie,1,Accès libre,24/7,false,FRSIGESIGE51911,2026-09-27,2026-09-27,QualiCharge


In [42]:
itinerance.loc[itinerance.id_station_itinerance == 'FRLDLPLFR4256EVCP', relations]

,index,contact_operateur,nom_enseigne,coordonneesXY,adresse_station,id_station_itinerance,nom_station,implantation_station,nbre_pdc,condition_acces,horaires,station_deux_roues,id_pdc_itinerance,date_maj,last_modified,datagouv_organization_or_owner
55092,55295,bornes@lidl.fr,LIDL,"[2.33401, 48.63200]",rue du Plessis,FRLDLPLFR4256EVCP,Saint-Geneviève-des-Bois - Plessis,Parking privé à usage public,8,Accès libre,24/7,false,FRLDLE00003929,2025-12-11,2025-12-11,Lidl
55093,55296,bornes@lidl.fr,LIDL,"[2.33401, 48.63200]",rue du Plessis,FRLDLPLFR4256EVCP,Saint-Geneviève-des-Bois - Plessis,Parking privé à usage public,8,Accès libre,24/7,false,FRLDLE00003930,2025-12-11,2025-12-11,Lidl
55094,55297,bornes@lidl.fr,LIDL,"[2.33401, 48.63200]",rue du Plessis,FRLDLPLFR4256EVCP,Saint-Geneviève-des-Bois - Plessis,Parking privé à usage public,8,Accès libre,24/7,false,FRLDLE00003931,2025-12-11,2025-12-11,Lidl
55095,55298,bornes@lidl.fr,LIDL,"[2.33401, 48.63200]",rue du Plessis,FRLDLPLFR4256EVCP,Saint-Geneviève-des-Bois - Plessis,Parking privé à usage public,8,Accès libre,24/7,false,FRLDLE00003932,2025-12-11,2025-12-11,Lidl
55100,55303,bornes@lidl.fr,LIDL,"[2.33401, 48.63200]",rue du Plessis,FRLDLPLFR4256EVCP,Saint-Geneviève-des-Bois - Plessis,Parking privé à usage public,8,Accès libre,24/7,false,FRLDLE00003963,2025-12-11,2025-12-11,Lidl
55101,55304,bornes@lidl.fr,LIDL,"[2.33401, 48.63200]",rue du Plessis,FRLDLPLFR4256EVCP,Saint-Geneviève-des-Bois - Plessis,Parking privé à usage public,8,Accès libre,24/7,false,FRLDLE00003964,2025-12-11,2025-12-11,Lidl
55102,55305,bornes@lidl.fr,LIDL,"[2.33401, 48.63200]",rue du Plessis,FRLDLPLFR4256EVCP,Saint-Geneviève-des-Bois - Plessis,Parking privé à usage public,8,Accès libre,24/7,false,FRLDLE00003965,2025-12-11,2025-12-11,Lidl
55103,55306,bornes@lidl.fr,LIDL,"[2.33401, 48.63200]",rue du Plessis,FRLDLPLFR4256EVCP,Saint-Geneviève-des-Bois - Plessis,Parking privé à usage public,8,Accès libre,24/7,false,FRLDLE00003966,2025-12-11,2025-12-11,Lidl
55416,55619,bornes@lidl.fr,LIDL,"[2.33427, 48.63166]",21 RUE DU PLESSIS,FRLDLPLFR4256EVCP,SAINTE GENEVIEVE DES BOIS,Parking privé à usage public,2,Accès libre,"Mo-Sa 08:30-20:30,Su 08:30-12:15",false,FRLDLE00007414,2025-12-11,2025-12-11,Lidl
55417,55620,bornes@lidl.fr,LIDL,"[2.33427, 48.63166]",21 RUE DU PLESSIS,FRLDLPLFR4256EVCP,SAINTE GENEVIEVE DES BOIS,Parking privé à usage public,2,Accès libre,"Mo-Sa 08:30-20:30,Su 08:30-12:15",false,FRLDLE00007415,2025-12-11,2025-12-11,Lidl


In [43]:
itinerance.loc[itinerance.id_station_itinerance == 'FRMBZESMYQC', relations]

,index,contact_operateur,nom_enseigne,coordonneesXY,adresse_station,id_station_itinerance,nom_station,implantation_station,nbre_pdc,condition_acces,horaires,station_deux_roues,id_pdc_itinerance,date_maj,last_modified,datagouv_organization_or_owner
70355,70732,support@mobilize-ps.fr,BODEMER - SCAUTO BAYEUX,"[-1.54360, 47.15431]","16 Bd Sadi Carnot, 14400 Bayeux",FRMBZESMYQC,Scauto Bayeux,Parking privé à usage public,1,Accès libre,"Mo-Fr 07:30-12:00,Mo-Fr 14:00-19:00",false,FRMBZESMYQC1,2024-06-03,2024-06-03,Mobilize Power Solutions
70702,71079,support@mobilize-ps.fr,BODEMER BAYEUX,"[-0.70022, 49.27084]","16 Bd Sadi Carnot, 14400 Bayeux",FRMBZESMYQC,BODEMER Bayeux,Parking privé à usage public,1,Accès libre,24/7,false,FRMBZESMYCQ1,2024-06-03,2024-06-03,Mobilize Power Solutions


In [44]:
itinerance.loc[itinerance.id_station_itinerance == 'FRIIMPIZIM54', relations]

,index,contact_operateur,nom_enseigne,coordonneesXY,adresse_station,id_station_itinerance,nom_station,implantation_station,nbre_pdc,condition_acces,horaires,station_deux_roues,id_pdc_itinerance,date_maj,last_modified,datagouv_organization_or_owner
72600,72979,sav@izivia.com,IZIVIA Impact,"[7.34832, 47.75053]",36 Rue du Printemps 68100 Mulhouse,FRIIMPIZIM54,m2A-MLH12-C-Mulhouse,Voirie,4,Accès libre,24/7,false,FRIIMEIZIM5412,2026-09-27,2026-09-27,IZIVIA
72601,72980,sav@izivia.com,IZIVIA Impact,"[7.34832, 47.75053]",36 Rue du Printemps 68100 Mulhouse,FRIIMPIZIM54,m2A-MLH12-C-Mulhouse,Voirie,4,Accès libre,24/7,false,FRIIMEIZIM5422,2026-09-27,2026-09-27,IZIVIA
105519,105900,sav@izivia.com,m2A,"[7.34832, 47.75053]","36 Rue du Printemps, 68100 Mulhouse",FRIIMPIZIM54,m2A-MLH12-C-Mulhouse,Voirie,2,Accès libre,24/7,false,FRIIMEIZIM5411,2026-09-27,2026-09-27,QualiCharge
105520,105901,sav@izivia.com,m2A,"[7.34832, 47.75053]","36 Rue du Printemps, 68100 Mulhouse",FRIIMPIZIM54,m2A-MLH12-C-Mulhouse,Voirie,2,Accès libre,24/7,false,FRIIMEIZIM5421,2026-09-27,2026-09-27,QualiCharge


In [45]:
itinerance.loc[itinerance.id_station_itinerance == 'FRDLM57724', relations]

,index,contact_operateur,nom_enseigne,coordonneesXY,adresse_station,id_station_itinerance,nom_station,implantation_station,nbre_pdc,condition_acces,horaires,station_deux_roues,id_pdc_itinerance,date_maj,last_modified,datagouv_organization_or_owner
164456,164837,mathieu.wiederkehr@orne-transition.fr,Delmonicos,"[6.10781, 49.26604]",6 Rue Abbé Alfred Sibile 57185 Vitry-sur-Orne,FRDLM57724,Parking de l'église,Voirie,1,Accès libre,24/7,false,FRDLME0CWEN8,2026-02-19,2026-02-19,SAS DELMONICOS
164470,164851,mathieu.wiederkehr@orne-transition.fr,Delmonicos,"[6.10554, 49.26906]",19 Rue Jean Jaurès 57185 Vitry-sur-Orne,FRDLM57724,Parking Mairie (Vitry-sur-Orne),Parking public,1,Accès libre,24/7,false,FRDLME59FI7N,2026-02-19,2026-02-19,SAS DELMONICOS
164475,164856,mathieu.wiederkehr@orne-transition.fr,Delmonicos,"[6.10554, 49.26906]",19 Rue Jean Jaurès 57185 Vitry-sur-Orne,FRDLM57724,Parking Mairie (Vitry-sur-Orne),Parking public,1,Accès libre,24/7,false,FRDLME8EYORD,2026-02-19,2026-02-19,SAS DELMONICOS
164480,164861,mathieu.wiederkehr@orne-transition.fr,Delmonicos,"[6.11352, 49.26579]",17 Rue de Gandrange 57185 Vitry-sur-Orne,FRDLM57724,Parking Rue de Gandrange (Vitry-sur-Orne),Parking public,1,Accès libre,24/7,false,FRDLMEAI8IEO,2026-02-19,2026-02-19,SAS DELMONICOS
164481,164862,mathieu.wiederkehr@orne-transition.fr,Delmonicos,"[6.11598, 49.27201]",5 Rue Louis Aragon 57185 Vitry-sur-Orne,FRDLM57724,Parking Collège (Vitry-sur-Orne),Parking public,1,Accès libre,24/7,false,FRDLMEAN3NAT,2026-02-19,2026-02-19,SAS DELMONICOS
164483,164864,mathieu.wiederkehr@orne-transition.fr,Delmonicos,"[6.11598, 49.27201]",5 Rue Louis Aragon 57185 Vitry-sur-Orne,FRDLM57724,Parking Collège (Vitry-sur-Orne),Parking public,1,Accès libre,24/7,false,FRDLMEAZLNJD,2026-02-19,2026-02-19,SAS DELMONICOS
164489,164870,mathieu.wiederkehr@orne-transition.fr,Delmonicos,"[6.10379, 49.27318]",61 Rue du 4 septembre 57185 Vitry-sur-Orne,FRDLM57724,Parking Rue du 4 septembre (Vitry-sur-Orne),Parking public,1,Accès libre,24/7,false,FRDLMECGEWSM,2026-02-19,2026-02-19,SAS DELMONICOS
164493,164874,mathieu.wiederkehr@orne-transition.fr,Delmonicos,"[6.10379, 49.27318]",61 Rue du 4 septembre 57185 Vitry-sur-Orne,FRDLM57724,Parking Rue du 4 septembre (Vitry-sur-Orne),Parking public,1,Accès libre,24/7,false,FRDLMEDWNGOG,2026-02-19,2026-02-19,SAS DELMONICOS
164505,164886,mathieu.wiederkehr@orne-transition.fr,Delmonicos,"[6.10554, 49.26906]",19 Rue Jean Jaurès 57185 Vitry-sur-Orne,FRDLM57724,Parking Mairie (Vitry-sur-Orne),Parking public,1,Accès libre,24/7,false,FRDLMEHICHKF,2026-02-19,2026-02-19,SAS DELMONICOS
164508,164889,mathieu.wiederkehr@orne-transition.fr,Delmonicos,"[6.11352, 49.26579]",17 Rue de Gandrange 57185 Vitry-sur-Orne,FRDLM57724,Parking Rue de Gandrange (Vitry-sur-Orne),Parking public,1,Accès libre,24/7,false,FRDLMEHWKEVZ,2026-02-19,2026-02-19,SAS DELMONICOS


### Repartition des defauts par operateur
Répartition pour les 10 opérateurs présentant le plus de défauts

In [ ]:
print('PdC avec une incohérence : ')
for operateur in oper_10:
    print('\n', sum(itinerance_5.contact_operateur == operateur))
    print('Répartition des incohérences ', operateur, ' :')
    for cont in controles:
        #print(cont, sum((itinerance_4['contact_operateur'] == 'supervision-ev.france@totalenergies.com') & ~(itinerance_4[cont])))
        print('{:<50} {:>5}'.format(cont, sum((itinerance_4['contact_operateur'] == operateur) & ~(itinerance_4[cont]))))

### Defauts operateur ALIZECHARGE
Les défauts identifiés concernent des stations utilisant un même identifiant. Par exemple, la station 'FRADPE77294001' contient 40 points de recharge avec plusieurs adresses, plusieurs localisations, plusieurs noms de station et un nombre de points de rechage variable.

In [ ]:
itinerance_4.loc[(~itinerance_4['nom_station - id_station_itinerance']) & (itinerance_4['contact_operateur'] == 'support@alizecharge.fr'), relations]

In [ ]:
itinerance_4.loc[(~itinerance_4['nbre_pdc - id_station_itinerance']) & (itinerance_4['contact_operateur'] == 'support@alizecharge.fr'), relations]

In [ ]:
itinerance_4.loc[itinerance_4.id_station_itinerance == 'FRADPE77294001', relations]

In [ ]:
itinerance_4.loc[itinerance_4.coordonneesXY == '[2.969121,48.282539]', relations]

### Defauts operateur LIDL
Les défauts identifiés concernent principalement l'affectation de coordonnées au point de recharge et non à la station. Par exemple, pour une station composée de quatre bornes de deux points de recharge, on trouve une coordonnée différente pour chacune des bornes.

In [ ]:
itinerance_4.loc[(~itinerance_4['coordonneesXY - id_station_itinerance']) & (itinerance_4['contact_operateur'] == 'bornes@lidl.fr') , relations].sort_values(by=["id_station_itinerance"])[0:50]

### Defauts operateur IZIVIA
Les défauts au nombre de 330 sont liés à une incohérence entre adresse et coordonnées (une même coordonnée a plusieurs adresses différentes). Ce défaut est lié soit à des erreurs de saisie, soit à des ajouts (ou remplacements) ultérieurs de stations à une même localisation.

Les stations Izivia en erreur ont un id_station_itinerance identique à l'id_pdc_itinerance, ce qui se traduit par des adresses multiples pour une même coordonnée.

Par exemple, sur un parking [-0.530598, 47.398424], on trouve 17 stations avec la même coordonnée, le même identifiant entre point de recharge et station et un nombre de PdC indiqué entre 1 et 3.

In [ ]:
itinerance_4.loc[itinerance_4['contact_operateur'] == 'sav@izivia.com', relations]

In [ ]:
itinerance_4.loc[(~itinerance_4['adresse_station - coordonneesXY']) & (itinerance_4['contact_operateur'] == 'sav@izivia.com') , relations].sort_values(by=["coordonneesXY"])[0:50]

In [ ]:
itinerance_4.loc[itinerance_4.coordonneesXY == '[-0.530598, 47.398424]', relations]

### Defauts operateur ENIplenitude
Les défauts (200 points de recharge) sont liés à une erreur de doumentation de l'implantation incohérence entre adresse et coordonnées (une même coordonnée a plusieurs adresses différentes). Ce défaut est lié soit à des erreurs de saisie, soit à des ajouts (ou remplacements) ultérieurs de stations à une même localisation.

Les stations Ionity en erreur ont un id_station_itinerance identique à l'id_pdc_itinerance ainsi qu'un ancien pdc de regroupement par station avec un nom d'enseigne différent (IONITY GMBH), ce qui se traduit par des adresses multiples pour une même coordonnée.

Par exemple, sur l'aire de Mornas [4.732409, 44.194832], on trouve 26 stations avec la même coordonnée, le même identifiant entre point de recharge et station et un nombre de PdC indiqué de 23 pour chaque station.

In [ ]:
itinerance_4.loc[itinerance_4['contact_operateur'] == 'support@service.emob.eniplenitude.com', relations]

In [ ]:
itinerance_4.loc[itinerance_4.id_station_itinerance == 'FRPLNEH002131', relations]

### Defauts operateur MOBILIZE
Les défauts (350 points de recharge) sont liés à une erreur de doumentation de l'implantation incohérence entre adresse et coordonnées (une même coordonnée a plusieurs adresses différentes). Ce défaut est lié soit à des erreurs de saisie, soit à des ajouts (ou remplacements) ultérieurs de stations à une même localisation.

In [ ]:
itinerance_4.loc[itinerance_4['contact_operateur'] == 'fastcharge.exploitation@mobilize.com', relations]

In [ ]:
itinerance_4.loc[itinerance_4.id_station_itinerance == 'FRMFCEALBERTVILLE', relations]

### Cohérence implantation_station - id_station
- 107 pdc sont liés à une erreur de choix d'implantation (couplé à d'autres erreurs)

In [ ]:
itinerance_4.loc[~itinerance_4['implantation_station - id_station_itinerance'], relations]

### Cohérence nom_station - id_station
- 210 pdc sont associés à une station avec un nom non cohérent

In [ ]:
itinerance_4.loc[~itinerance_4['nom_station - id_station_itinerance'], relations]

### Coherence adresse - coordonnees
- 2337 pdc ont une adresse non cohérente avec les coordonnées géographiques

In [ ]:
itinerance_4.loc[~itinerance_4['adresse_station - coordonneesXY'], relations]

In [ ]:
itinerance_4.loc[itinerance_4.coordonneesXY == '[5.2592028, 46.1746523]', relations]

In [ ]:
itinerance_4.loc[itinerance_4.adresse_station == 'RN 75 ZI DE NOIREFONTAINE 01250 Montagnat', relations]

In [ ]:
itinerance_4.loc[itinerance_4.adresse_station == ' RN 75 ZI DE NOIREFONTAINE 01250 Montagnat', relations]

### Cohérence station - coordonnées
- 218 pdc sont associés à des stations avec plusieurs coordonnées (majorité AlizeCharge)

In [ ]:
itinerance_4.loc[~itinerance_4['coordonneesXY - id_station_itinerance'], relations]

In [ ]:
itinerance_5.loc[itinerance_5.id_station_itinerance	 == 'FRTLSE31555040', relations]

In [ ]:
itinerance_5.loc[itinerance_5.id_station_itinerance	 == 'FRFR1PEHNKRRPQNR', relations]

- exemple : une station avec plusieurs coordonnées -> incompréhension distinction station / pdc

In [ ]:
itinerance_5.loc[itinerance_5.id_station_itinerance == 'FR073PCAMAIEUFR', relations]

- exemple : une station avec plusieurs coordonnées -> regroupement de stations ?

In [ ]:
itinerance_5.loc[itinerance_5.id_station_itinerance == 'FR55CPBP514', relations]

- exemple : plusieurs stations d'opérateurs différents avec les mêmes coordonnées -> suppression non effectuée lors d'un changement d'opérateur ?

In [ ]:
itinerance_4.loc[itinerance_4.coordonneesXY == '[0.08745032, 48.40887156]', relations]

In [ ]:
itinerance_4.loc[itinerance_4.coordonneesXY == '[-4.374469, 48.471878]', relations]

- exemple : quatre stations avec une localisation identique mais deux adresses différentes + id identique entre station et pdc -> erreur d'adresse et erreur d'id_station ?

In [ ]:
itinerance_5.loc[itinerance_5.coordonneesXY	 == '[-0.366184, 43.332508]', relations]

### Incoherence nbre_pdc
- 583 pdc ont un champ 'nbre_pdc' mal documenté

In [ ]:
itinerance_4.loc[~itinerance_4['nbre_pdc - id_station_itinerance'], relations][200:215]

### coherence station - enseigne
- 5 stations ont plusieurs noms d'enseigne

In [ ]:
itinerance_4.loc[~itinerance_4['nom_enseigne - id_station_itinerance'], relations]

In [ ]:
itinerance_init.loc[itinerance_init.id_station_itinerance == 'FRFR1EAYRU', relations]

In [ ]:
itinerance_init.loc[itinerance_init.nom_enseigne == 'RME Loos (FR-FR1)', relations]

### cohérence station - horaires
- exemple de station avec plusieurs types d'horaire (15)

In [ ]:
itinerance_init.loc[itinerance_init.id_station_itinerance == 'FRS23D2302001', relations]

### cohérence station - deux-roues
- exemple de station avec plusieurs indications d'accessibilité deux-roues (583)

In [ ]:
itinerance_init.loc[itinerance_init.id_station_itinerance == 'FRV75PPX1716', relations]

### stations avec pdc > 350 kW

- exemple de station avec une puissance de plus de 350 kW

In [ ]:
data['p_nom'] = data['puissance_nominale'].str.replace(",", ".")

In [ ]:
data['p_nom'] = data['p_nom'].astype('float')

In [ ]:
from copy import copy
data350 = copy(data.loc[data["p_nom"] > 350, :])

In [ ]:
data350['index'] = 0

In [ ]:
data350.loc[:, relations]

In [ ]:
data350['contact_operateur'].unique()